# syft-ingest API Exploration

In [1]:
import syft_ingest as si
from syft_ingest import async_gather, gather

## 1. Simplified gather() API — YouTube video

In [ ]:
# Simplest case: just platform + URLs
corpus = si.gather(
    "youtube", ["https://www.youtube.com/watch?v=zY2dAK-pMPI&t=11s"]
)  # Andrew Trask: talk

print(f"Items fetched: {len(corpus.all_items())}")
if corpus.all_items():
    item = corpus.all_items()[0]
    print(f"  Title: {item.title}")
    print(f"  Author: {item.author}")

In [ ]:
corpus.export("./data/youtube.jsonl")

## 2. Channel enumeration with config

In [ ]:
# With config options passed as kwargs
corpus = si.gather(
    "youtube",
    ["https://www.youtube.com/@iamtrask"],
    playlistend=3,  # Cap at 3 videos to keep it fast
)

print(f"Videos found: {len(corpus.all_items())}")
for item in corpus.all_items():
    print(f"  [{item.published_at}] {item.title}")

In [ ]:
corpus.export("./data/youtube2.jsonl")

## 3. Facebook scraping

Requires `BRIGHTDATA_API_TOKEN`. Triggers a real scrape job and polls until done (~60-120s).

In [2]:
import os

token = os.getenv("BRIGHTDATA_API_TOKEN")
print(
    f"BRIGHTDATA_API_TOKEN : {token[:3] + '...' + token[-3:] if token else 'NOT SET -- BrightData cells will fail'}"
)

BRIGHTDATA_API_TOKEN : 7a5...1d5


In [5]:
corpus = await async_gather(
    "facebook",
    ["https://www.facebook.com/profile.php?id=61583734012155"],
    author="Syft Influencer Test",
    post_limit=5,
)

print(f"Items fetched: {len(corpus.all_items())}")
print(f"Author: {corpus.person}")

if corpus.all_items():
    item = corpus.all_items()[0]
    print("\nFirst item:")
    print(f"  Type: {type(item).__name__}")
    print(f"  Title: {item.title}")
    print(f"  Text: {(item.text or '')[:120]}")

2026-04-13 21:16:31.917 | INFO     | syft_ingest.sources.brightdata:fetch_async:116 - Fetching 1 URL(s) for facebook
2026-04-13 21:16:31.918 | DEBUG    | syft_ingest.sources.brightdata:fetch_async:130 - Using platform scraper: facebook
2026-04-13 21:16:32.263 | INFO     | syft_ingest.sources.brightdata:fetch_async:152 - Triggering facebook scrape for https://www.facebook.com/profile.php?id=61583734012155
2026-04-13 21:16:32.668 | DEBUG    | syft_ingest.sources.brightdata:fetch_async:158 - Scrape job created: sd_mny1v9xx2cezhsm7sc
2026-04-13 21:16:32.670 | INFO     | syft_ingest.sources.brightdata:fetch_async:161 - Polling job sd_mny1v9xx2cezhsm7sc with timeout=180s, poll_interval=5s
2026-04-13 21:18:25.318 | DEBUG    | syft_ingest.sources.brightdata:fetch_async:172 - Job sd_mny1v9xx2cezhsm7sc completed
2026-04-13 21:18:26.143 | DEBUG    | syft_ingest.sources.brightdata:fetch_async:176 - Fetched 162690 bytes from job
2026-04-13 21:18:26.144 | WARNING  | syft_ingest.sources.brightdata:_p

Items fetched: 25
Author: Syft Influencer Test

First item:
  Type: SocialPostResult
  Title: 122103859383124467
  Text: Private Deep Learning for Hospitals using Federated Learning and Differential Privacy

Today, Kritika Prakash, Lucile Sa


In [7]:
corpus.export("data/fb.jsonl")

2026-04-13 21:19:24.579 | INFO     | syft_ingest.core.exporters:_export_jsonl:74 - Exported 25 items to data/fb.jsonl


## 6. Instagram scraping with posts_limit for testing

In [8]:
corpus = await async_gather(
    "instagram",
    ["https://www.instagram.com/syft_influencer_test/"],
    author="Syft Influencer Test",
    timeout=120,
    poll_interval=5,
    posts_limit=5,  # Limit to 5 posts for quick testing (no posts_limit = all posts)
)

print(f"Items fetched: {len(corpus.all_items())}")
for i, item in enumerate(corpus.all_items()[:3], 1):
    print(f"\n{i}. {type(item).__name__}: {item.title}")

2026-04-13 21:23:52.987 | INFO     | syft_ingest.sources.brightdata:fetch_async:116 - Fetching 1 URL(s) for instagram
2026-04-13 21:23:52.988 | DEBUG    | syft_ingest.sources.brightdata:fetch_async:130 - Using platform scraper: instagram
2026-04-13 21:23:53.401 | INFO     | syft_ingest.sources.brightdata:fetch_async:152 - Triggering instagram scrape for https://www.instagram.com/syft_influencer_test/
2026-04-13 21:23:53.810 | DEBUG    | syft_ingest.sources.brightdata:fetch_async:158 - Scrape job created: sd_mny24qbh16a5m5tzx7
2026-04-13 21:23:53.811 | INFO     | syft_ingest.sources.brightdata:fetch_async:161 - Polling job sd_mny24qbh16a5m5tzx7 with timeout=120s, poll_interval=5s
2026-04-13 21:23:59.477 | DEBUG    | syft_ingest.sources.brightdata:fetch_async:172 - Job sd_mny24qbh16a5m5tzx7 completed
2026-04-13 21:23:59.991 | DEBUG    | syft_ingest.sources.brightdata:fetch_async:176 - Fetched 22807 bytes from job
2026-04-13 21:23:59.992 | INFO     | syft_ingest.sources.brightdata:_parse_

FetchEmptyResultError: No content items found in instagram response

## 7. Error handling and validation

In [9]:
# Invalid platform raises ValueError
try:
    corpus = gather("tiktok", ["https://tiktok.com/user"])
except KeyError as e:
    print(f"✅ ValueError caught for unsupported platform: {e}")

# Missing URLs raises ValueError
try:
    corpus = gather("youtube", None)
except ValueError as e:
    print(f"✅ ValueError caught for missing URLs: {e}")

2026-04-13 21:25:49.226 | ERROR    | syft_ingest.core.gather:gather:83 - No fetcher registered for tiktok: "No fetcher registered for platform 'tiktok' and extractor 'brightdata'. Register one with register_fetcher()."
2026-04-13 21:25:49.227 | ERROR    | syft_ingest.core.gather:gather:80 - Invalid platform: Platform 'youtube' requires urls list


✅ ValueError caught for unsupported platform: "No fetcher registered for platform 'tiktok' and extractor 'brightdata'. Register one with register_fetcher()."
✅ ValueError caught for missing URLs: Platform 'youtube' requires urls list
